# 01 — Análise exploratória do tabuleiro

Este notebook explora a configuração selecionada do jogo. As métricas são descritivas: aluguel/preço não é retorno esperado, porque ainda não considera probabilidade de visita, propriedade dos terrenos, construção ou decisões dos jogadores.

**Execute a partir da raiz do repositório** para que os imports `app.*` sejam resolvidos.

## 1. Imports e carregamento da configuração

In [4]:
from pathlib import Path
import sys

current = Path.cwd().resolve()

project_root = next(
    (
        path
        for path in (current, *current.parents)
        if (path / "app" / "main.py").is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Não encontrei app/main.py nos diretórios atuais ou superiores. "
        "Verifique se o notebook está dentro do repositório MONOPOLY-AI."
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Raiz do projeto:", project_root)

Raiz do projeto: /media/marth/Dados/Arquivos/Monopoly-AI


In [ ]:
from pathlib import Path
import sys

print("Diretório atual:", Path.cwd())
print("Python utilizado:", sys.executable)
print("Diretórios de importação:")
print(*sys.path, sep="\n")

ModuleNotFoundError: No module named 'actions'

## 2. Visão estrutural

In [ ]:
summary = analyze_board(game_config)
summary_df = pd.DataFrame([
    {'metric': key, 'value': value}
    for key, value in summary.items()
    if key not in {'space_type_counts', 'property_groups'}
])
display(summary_df)
display(pd.DataFrame([
    {'space_type': space_type, 'count': count}
    for space_type, count in summary['space_type_counts'].items()
]).sort_values('count', ascending=False))

## 3. Distribuição dos tipos de casa

In [ ]:
type_counts = spaces_df['type'].value_counts().sort_values()
ax = type_counts.plot(kind='barh', figsize=(9, 5))
ax.set_title('Composição do tabuleiro por tipo de casa')
ax.set_xlabel('Quantidade de casas')
ax.set_ylabel('Tipo de casa')
plt.tight_layout()
plt.show()

## 4. Preço de compra por propriedade

In [ ]:
plot_df = properties_df.sort_values('price', ascending=True)
fig, ax = plt.subplots(figsize=(11, max(6, len(plot_df) * 0.28)))
ax.barh(plot_df['name'], plot_df['price'])
ax.set_title('Preço de compra das propriedades')
ax.set_xlabel('Preço')
ax.set_ylabel('Propriedade')
ax.grid(axis='x', alpha=0.25)
plt.tight_layout()
plt.show()

## 5. Aluguel-base e aluguel/preço

A razão aluguel-base/preço é apenas uma comparação estática. Não representa rentabilidade anual nem retorno esperado por turno.

In [ ]:
plot_df = properties_df.sort_values('rent_base', ascending=True)
fig, ax = plt.subplots(figsize=(11, max(6, len(plot_df) * 0.28)))
ax.barh(plot_df['name'], plot_df['rent_base'])
ax.set_title('Aluguel-base por propriedade')
ax.set_xlabel('Aluguel-base')
ax.set_ylabel('Propriedade')
ax.grid(axis='x', alpha=0.25)
plt.tight_layout()
plt.show()

display(properties_df[['name', 'color_group', 'price', 'rent_base', 'base_rent_to_price']]
        .sort_values('base_rent_to_price', ascending=False)
        .style.format({'price': '{:,.0f}', 'rent_base': '{:,.0f}', 'base_rent_to_price': '{:.2%}'}))

## 6. Comparação entre grupos de cor

In [ ]:
display(groups_df)

plot_groups = groups_df.sort_values('total_purchase_cost')
fig, ax = plt.subplots(figsize=(10, max(4, len(plot_groups) * 0.4)))
ax.barh(plot_groups['color_group'], plot_groups['total_purchase_cost'])
ax.set_title('Custo de compra do conjunto de cada grupo')
ax.set_xlabel('Soma dos preços de compra')
ax.set_ylabel('Grupo de cor')
ax.grid(axis='x', alpha=0.25)
plt.tight_layout()
plt.show()

## 7. Relação entre preço e aluguel-base

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
for group, subset in properties_df.groupby('color_group', dropna=False):
    ax.scatter(subset['price'], subset['rent_base'], label=str(group), s=65)
    for _, row in subset.iterrows():
        ax.annotate(row['name'], (row['price'], row['rent_base']), xytext=(4, 4),
                    textcoords='offset points', fontsize=8)
ax.set_title('Preço de compra versus aluguel-base')
ax.set_xlabel('Preço de compra')
ax.set_ylabel('Aluguel-base')
ax.grid(alpha=0.25)
ax.legend(title='Grupo de cor', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 8. Inspeção dos dados e limitações

A tabela abaixo ajuda a conferir se a configuração carregada contém os campos esperados. Antes de interpretar resultados, verifique se os nomes e valores correspondem à variante escolhida.

Próxima etapa: integrar as probabilidades de visita do módulo de Markov. Só depois disso poderemos construir estimativas de receita de aluguel sob hipóteses explícitas.

In [ ]:
display(spaces_df.head(12))
display(properties_df)